Simulación de Procesos Financieros
Maria Fernanda Gómez Flores
Capital inicial: $100,000 MXN. Horizonte: 1 año. Simulaciones: 10,000.
Se utilizan rendimientos logarítmicos y se simula el tipo de cambio USD/MXN.

1. Descargar datos históricos (3 años)

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
tickers = ['NVDA', 'BIMBOA.MX', 'CEMEXCPO.MX', 'AZN', 'MXN=X']
df = yf.download(tickers, period='3y')['Close']
df.head()

/var/folders/x9/60hcc2ln62d3r3xyv90d3wl40000gn/T/ipykernel_39711/2989473120.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tickers, period='3y')['Close']
[*********************100%***********************]  5 of 5 completed


Ticker,AZN,BIMBOA.MX,CEMEXCPO.MX,MXN=X,NVDA
Date,,,,,
2023-10-30,NaN,66.687401,10.555708,NaN,NaN
2023-10-31,NaN,69.706947,10.585139,NaN,NaN
2023-11-01,NaN,72.250237,10.742101,NaN,NaN
2023-11-03,121.333717,74.936394,11.458242,17.52700,44.870991
2023-11-06,121.504456,78.346489,11.615203,17.47213,45.614765


Agregamos la fila del 2 de noviembre de 2026 con los precios del pizarrón.

In [3]:
df.loc[pd.Timestamp('2026-11-02'), ['AZN', 'BIMBOA.MX', 'CEMEXCPO.MX', 'NVDA', 'MXN=X']] = [186.58, 66.18, 20.23, 202.06, 18.58]
df = df.sort_index()
df.tail()

Ticker,AZN,BIMBOA.MX,CEMEXCPO.MX,MXN=X,NVDA
Date,,,,,
2026-10-28,169.509995,61.830002,19.700001,18.687040,215.380005
2026-10-29,174.399994,63.689999,20.480000,18.631790,213.660004
2026-10-30,186.649994,66.180000,20.230000,18.852100,204.639999
2026-11-02,186.580000,66.180000,20.230000,18.580000,202.060000
2026-11-03,NaN,NaN,NaN,18.572359,NaN


2. Estimar mu y sigma con rendimientos logarítmicos

In [4]:
rets = np.log(df / df.shift(1))
rets = rets.dropna()
rets.head()

Ticker,AZN,BIMBOA.MX,CEMEXCPO.MX,MXN=X,NVDA
Date,,,,,
2023-11-06,0.001406,0.044501,0.013606,-0.003136,0.016440
2023-11-07,-0.006893,-0.018034,0.007573,0.003609,0.004449
2023-11-08,-0.001259,-0.009203,0.017450,-0.003719,0.013380
2023-11-09,0.008776,0.008088,0.012280,0.003143,0.008041
2023-11-10,-0.014459,-0.002357,-0.000814,0.014961,0.029073


In [5]:
mu = np.mean(rets, axis=0)
sigma = np.std(rets, axis=0, ddof=1)

print(mu)
print(sigma)

Ticker
AZN            0.000603
BIMBOA.MX     -0.000231
CEMEXCPO.MX    0.000758
MXN=X          0.000307
NVDA           0.002774
dtype: float64
Ticker
AZN            0.015813
BIMBOA.MX      0.018745
CEMEXCPO.MX    0.020789
MXN=X          0.006936
NVDA           0.029501
dtype: float64


3. Simular T = 1 año, M = 10,000 escenarios

Como mu y sigma son diarios, usamos mu*252 y sigma*np.sqrt(252) para simular un año. El tipo de cambio también se simula en cada escenario.

In [6]:
T = 1
M = 10000
S0 = df.loc[pd.Timestamp('2026-11-02')]

np.random.seed(32)
precios = []

for i in range(M):
    ST = []
    for ticker in tickers:
        Z = np.random.normal(0, 1)
        ST.append(S0[ticker] * np.exp(mu[ticker] * 252 * T + sigma[ticker] * np.sqrt(252 * T) * Z))
    precios.append(ST)

precios = pd.DataFrame(precios, columns=tickers)
precios.head()

,NVDA,BIMBOA.MX,CEMEXCPO.MX,AZN,MXN=X
0,345.233725,83.678781,29.660376,221.054737,21.870987
1,533.873874,96.758724,42.392944,203.405853,18.610540
2,293.585569,111.237415,44.430335,243.548392,18.845324
3,428.842326,97.969440,27.494218,212.393737,22.209311
4,515.130548,48.993276,27.841366,195.212377,20.164336


Convertimos NVDA y AZN a pesos utilizando el tipo de cambio simulado de cada escenario.

In [7]:
precios['NVDA_MXN'] = precios['NVDA'] * precios['MXN=X']
precios['AZN_MXN'] = precios['AZN'] * precios['MXN=X']
precios.head()

,NVDA,BIMBOA.MX,CEMEXCPO.MX,AZN,MXN=X,NVDA_MXN,AZN_MXN
0,345.233725,83.678781,29.660376,221.054737,21.870987,7550.602462,4834.685380
1,533.873874,96.758724,42.392944,203.405853,18.610540,9935.680900,3785.492697
2,293.585569,111.237415,44.430335,243.548392,18.845324,5532.715232,4589.748416
3,428.842326,97.969440,27.494218,212.393737,22.209311,9524.292484,4717.118509
4,515.130548,48.993276,27.841366,195.212377,20.164336,10387.265553,3936.327996


4. Calcular N (número de acciones)

Usamos 25% para cada acción. Para calcular N, convertimos los precios iniciales en dólares con el tipo de cambio inicial de 18.58 MXN/USD.

In [8]:
V0 = 100000
w = [0.25, 0.25, 0.25, 0.25]
acciones = ['NVDA', 'BIMBOA.MX', 'CEMEXCPO.MX', 'AZN']

N = {}
for i in range(len(acciones)):
    ticker = acciones[i]
    if ticker == 'NVDA' or ticker == 'AZN':
        precio_mxn = S0[ticker] * S0['MXN=X']
    else:
        precio_mxn = S0[ticker]
    N[ticker] = int(w[i] * V0 / precio_mxn)

N

{'NVDA': 6, 'BIMBOA.MX': 377, 'CEMEXCPO.MX': 1235, 'AZN': 7}

Hasta aquí llegamos al paso 4. Los pasos 5 y 6 se harán después.